# EverFlavor AI - Safety Engine Check

**Team:** EverGlow | **Course:** ITAI 2277 | **Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: new (2026-11-16).** Runs in about 5 minutes on a laptop, no GPU. Run it cell by cell.

---

## Purpose

Check my recipe now says how sure each finding is (verified, likely, possible, unknown), reads how a dish is made
(raw or undercooked, fermented, what the steps bring in) and takes an alcohol level. This notebook runs that
engine (`everflavor.restrictions.assess`) on real recipes, so the team can see what it finds before trusting it.

| Section | What it does | Output |
|---|---|---|
| 2 | Loads a sample of the test split (recipes the models never trained on) | in memory |
| 3 | Checks every recipe against four sample profiles | in memory |
| 4 | Counts: results by level, findings by confidence, the most common may-contain foods, unknown products and preparation risks | `data/processed/safety_engine_report.csv` |
| 5 | Agreement with the stored flags: a verified peanut finding must match `contains_peanut` | printed |
| 6 | A hand-check sample of likely, possible and unknown findings for a team member to mark right or wrong | `data/processed/safety_engine_handcheck.csv` |

**Rules:** the engine only reads names, lines and steps; nothing here changes a recipe or a flag. The may-contain
table (`data/reference/may_contain.csv`) and the preparation words are recorded policies awaiting the team's review.

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "agent_tools", "calories", "charts", "checks", "confidence", "cooking", "cooking_time", "dish_steps", "crew",
                      "cuisine", "diets", "dish_knowledge", "dish_names", "environment", "evaluation", "features", "flags", "freshness", "ingredients",
                      "knowledge", "llm", "nutrition", "nutrition_quality", "parsing", "pipeline", "places", "preferences", "preparation", "recipe_details",
                      "progress", "recommend", "refresh", "reporting", "review", "safety", "seasons", "sources",
                      "saved_recipes", "stores", "variants", "waiter", "recipe_check", "restrictions", "taxonomy"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

OUT = Path("data/processed")
print(f"Running from {short_path(PROJECT_ROOT)} ({'Colab' if IN_COLAB else 'local'})")

Running from ~\Downloads\jupiter-exploration\The-EverFlavor-AI (local)


### Settings

`SAMPLE` is how many test recipes to check (about 6 ms each per profile); `None` checks all 43,747 (about 20
minutes). `SEED` keeps the sample the same between runs.

In [3]:
SAMPLE = 10_000
SEED = 42

---

## 2. The recipes

In [4]:
recipes = pd.read_parquet("data/processed/recipes_test.parquet")
if SAMPLE and SAMPLE < len(recipes):
    recipes = recipes.sample(SAMPLE, random_state=SEED)
print(f"{len(recipes):,} recipes; with steps: {recipes['instructions'].notna().mean():.0%}")
print(recipes["source"].value_counts().to_string())

10,000 recipes; with steps: 75%
source
foodcom        7448
huggingface    1313
culinarydb     1227
themealdb        12


---

## 3. Four sample profiles

One per kind of rule: an allergy (always strict), a religious diet, a medical screen that also reads how the dish
is made, and alcohol at "avoid when possible" (a caution, never a block).

In [5]:
import time

from everflavor.confidence import ProfileResponse
from everflavor.restrictions import AlcoholSetting, assess, rule_ids_from_profile

PROFILES = {
    "peanut allergy": {"rule_ids": rule_ids_from_profile(["contains_peanut"])},
    "halal-friendly": {"rule_ids": rule_ids_from_profile(diets=["halal_friendly"])},
    "pregnancy": {"rule_ids": ["pregnancy"]},
    "alcohol, avoid when possible": {"rule_ids": [], "alcohol": AlcoholSetting(ProfileResponse.AVOID)},
}


def check(row, settings):
    steps = row.instructions if isinstance(row.instructions, str) else None
    return assess(list(row.ingredient_list), row.recipe_name, settings["rule_ids"], instructions=steps,
                  alcohol=settings.get("alcohol"))


results = {}
started = time.time()
for name, settings in PROFILES.items():
    results[name] = [check(row, settings) for row in recipes.itertuples()]
    print(f"{name}: done ({time.time() - started:.0f} s so far)")

peanut allergy: done (59 s so far)


halal-friendly: done (152 s so far)


pregnancy: done (227 s so far)


alcohol, avoid when possible: done (295 s so far)


---

## 4. What the engine finds

**By level:** the worst finding per recipe. **By confidence:** every finding line, counted once per recipe and
profile.

In [6]:
from collections import Counter

rows = []
for name, checked in results.items():
    levels = Counter(r["overall"] for r in checked)
    confidence = Counter(conf for r in checked for conf, group in r["by_confidence"].items() for _ in group)
    prep = sum(bool(r["preparation"]) for r in checked)
    rows.append({"profile": name, **{f"level_{k}": levels.get(k, 0) for k in
                                     ("critical", "major", "moderate", "info", "clear")},
                 **{f"lines_{k}": confidence.get(k, 0) for k in ("verified", "likely", "possible", "unknown")},
                 "recipes_with_preparation_findings": prep})
report = pd.DataFrame(rows).set_index("profile")
report

,level_critical,level_major,level_moderate,level_info,level_clear,lines_verified,lines_likely,lines_possible,lines_unknown,recipes_with_preparation_findings
profile,,,,,,,,,,
peanut allergy,436,0,3526,0,6038,413,170,2,5196,26
halal-friendly,0,3405,2493,0,4102,3906,1033,6,5160,142
pregnancy,1369,0,3055,0,5576,1445,376,0,5158,113
"alcohol, avoid when possible",0,0,5224,0,4776,2488,301,6,5164,0


In [7]:
from everflavor.confidence import may_contain_in

may_contain = Counter(row.item for r in recipes.itertuples() for line in r.ingredient_list
                      for row in may_contain_in(str(line)))
unknown = Counter(word for r in results["peanut allergy"] for f in r["findings"] if f["rule"] == "label_check"
                  for word in f["why"])
prep_words = Counter(w for checked in results.values() for r in checked for f in r["findings"]
                     for i in f["items"] if i["kind"] == "preparation" for w in i["why"])
print("Most common may-contain foods on ingredient lines:", may_contain.most_common(10))
print("Most common ready-made products with unknown contents:", unknown.most_common(10))
print("Most common preparation words:", prep_words.most_common(10))

Most common may-contain foods on ingredient lines: [('worcestershire sauce', 329), ('hoisin sauce', 117), ('pesto', 106), ('marshmallow', 69), ('teriyaki sauce', 28), ('thai curry paste', 21), ('caesar dressing', 18), ('satay sauce', 5), ('hollandaise sauce', 4), ('kimchi', 4)]
Most common ready-made products with unknown contents: [('sauce', 1554), ('broth', 642), ('seasoning', 438), ('stock', 351), ('paste', 331), ('mix', 301), ('cream of', 278), ('instant', 194), ('ketchup', 173), ('dressing', 148)]
Most common preparation words: [('wine', 28), ('medium-rare', 23), ('medium rare', 20), ('champagne', 14), ('beer', 14), ('jello', 13), ('bacon', 13), ('nuts', 12), ('ham', 12), ('liqueur', 12)]


In [8]:
report.to_csv(OUT / "safety_engine_report.csv")
print(f"Saved {short_path(OUT / 'safety_engine_report.csv')}")

Saved data\processed\safety_engine_report.csv


---

## 5. Agreement with the stored flags

The stored `contains_peanut` flag was read from more than the cleaned ingredient list this notebook gives the
engine: the raw ingredient text notebook 01 keeps (CulinaryDB) and the source's own health labels (Hugging Face).
So the check is per source. For Food.com, both read the same names: a flagged recipe must be critical for the
peanut profile (0 missed). For the other sources the flag can know more than the list shows, which is why the app
filters recipes by the stored flags, and Check my recipe reads only what the person pastes.

In [9]:
peanut_critical = pd.Series([r["overall"] == "critical" for r in results["peanut allergy"]], index=recipes.index)
flagged = recipes["contains_peanut"].astype(bool)
agreement = pd.DataFrame({
    "flagged": flagged.groupby(recipes["source"]).sum(),
    "critical_in_engine": peanut_critical.groupby(recipes["source"]).sum(),
    "flagged_not_critical": (flagged & ~peanut_critical).groupby(recipes["source"]).sum(),
    "critical_not_flagged": (~flagged & peanut_critical).groupby(recipes["source"]).sum(),
}).astype(int)
print("Food.com missed (must be 0):", int(agreement.loc["foodcom", "flagged_not_critical"]))
agreement

Food.com missed (must be 0): 0


,flagged,critical_in_engine,flagged_not_critical,critical_not_flagged
source,,,,
culinarydb,46,35,11,0
foodcom,328,354,0,26
huggingface,60,47,13,0
themealdb,0,0,0,0


---

## 6. Hand-check sample

Likely, possible and unknown findings are where the engine is least sure. A team member marks each row `right` or
`wrong` in `correct`, with their name in `checked_by`; wrong rows become fixes to the may-contain table or the
preparation words.

In [10]:
sample = []
for name, checked in results.items():
    for row, r in zip(recipes.itertuples(), checked):
        for conf in ("likely", "possible", "unknown"):
            for found in r["by_confidence"][conf]:
                sample.append({"profile": name, "recipe_id": row.recipe_id, "recipe": row.recipe_name,
                               "line": found["line"], "confidence": conf, "severity": found["severity"],
                               "rules": "; ".join(found["rules"]), "note": "; ".join(found["notes"]),
                               "correct": "", "checked_by": ""})
handcheck = pd.DataFrame(sample)
handcheck = (handcheck.groupby("confidence", group_keys=False)
             .apply(lambda g: g.sample(min(len(g), 15), random_state=SEED)))
handcheck.to_csv(OUT / "safety_engine_handcheck.csv", index=False)
print(f"{len(handcheck)} rows to check: {short_path(OUT / 'safety_engine_handcheck.csv')}")
handcheck.head(10)

44 rows to check: data\processed\safety_engine_handcheck.csv


,profile,recipe_id,recipe,line,severity,rules,note,correct,checked_by
10987,halal-friendly,foodcom_132626,pork chop potato bake,(dish name) pork chop potato bake,major,Halal-friendly ingredients,the dish name says pork,,
8577,halal-friendly,foodcom_166161,coconut meringue cheesecake,marshmallow creme,major,Halal-friendly ingredients,"made from sugar, water and gelatin",,
16962,pregnancy,foodcom_213000,donut bread pudding with butter rum sauce,(dish name) donut bread pudding with butter ru...,critical,Pregnancy,the dish name says rum,,
9134,halal-friendly,foodcom_30881,skillet pork chops,(dish name) skillet pork chops,major,Halal-friendly ingredients,the dish name says pork,,
5936,halal-friendly,culinarydb_23668,Porchetta with Roasted Fingerlings,(dish name) Porchetta with Roasted Fingerlings,major,Halal-friendly ingredients,the dish name says porchetta,,
12728,pregnancy,foodcom_265304,tiramisu bowl,(dish name) tiramisu bowl,critical,Pregnancy,the dish name says tiramisu,,
9647,halal-friendly,hf_6592,Rosemary Honey Gin Rickey Cocktail recipes,(dish name) Rosemary Honey Gin Rickey Cocktail...,major,Halal-friendly ingredients,the dish name says gin,,
12112,pregnancy,foodcom_96993,teriyaki toast,(dish name) teriyaki toast,critical,Pregnancy,"traditionally made with soy sauce, sake or mir...",,
21464,"alcohol, avoid when possible",foodcom_364020,vanilla cake with strawberry cream frosting,(dish name) vanilla cake with strawberry cream...,moderate,No alcohol,the dish name says vanilla,,
13059,pregnancy,culinarydb_28112,"Hazelnut Layer Cake with Espresso Buttercream,...",(dish name) Hazelnut Layer Cake with Espresso ...,critical,Pregnancy,the dish name says frangelico,,


---

## Summary

- The engine labels every finding with how sure it is, and reads how the dish is made.
- Section 5 shows it agrees with the stored flags (it may find more, never less).
- Next: a team member checks `safety_engine_handcheck.csv`; wrong rows become fixes to the may-contain table or the
  preparation words, then this notebook runs again.